# Text Classification with Neural Networks

This notebook provides a comprehensive guide to text classification using neural networks. We'll cover the entire workflow from preprocessing text data to building, training, evaluating, and deploying neural network models for text classification tasks.

## 1. Import Required Libraries

First, let's import all the necessary libraries for our text classification project.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle
import os
from collections import Counter

# Text processing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Deep learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras.layers import Dense, Embedding, LSTM, GRU, Bidirectional
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dropout, Input, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# For visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Download necessary NLTK resources
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')

## 2. Data Loading and Exploration

In this section, we'll load a text classification dataset and explore its characteristics. We'll use the IMDB movie reviews dataset, which is a popular benchmark for sentiment analysis.

In [ ]:
# Load IMDB dataset from Keras
from tensorflow.keras.datasets import imdb

# Load the data with the top 10,000 words
num_words = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

# Get the word index dictionary
word_index = imdb.get_word_index()

# Flip the word index to get a dictionary mapping integers to words
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'
reverse_word_index[3] = '<UNUSED>'

# Function to convert a review (list of integers) back to words
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded_review])

# Print basic dataset information
print(f"Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}, Test labels shape: {y_test.shape}")

# Print a sample review
print("\nSample review (decoded):")
print(decode_review(X_train[0]))
print("\nLabel:", y_train[0])  # 0 = negative, 1 = positive

# Check class distribution
print("\nClass distribution:")
print("Training set:", Counter(y_train))
print("Test set:", Counter(y_test))

# Visualize sequence lengths
train_lengths = [len(x) for x in X_train]
test_lengths = [len(x) for x in X_test]

plt.figure(figsize=(12, 6))
plt.hist(train_lengths, bins=50, alpha=0.5, label='Training set')
plt.hist(test_lengths, bins=50, alpha=0.5, label='Test set')
plt.xlabel('Review Length (words)')
plt.ylabel('Frequency')
plt.title('Distribution of Review Lengths')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Calculate statistics on sequence lengths
print("\nSequence length statistics:")
print(f"Training set - Mean: {np.mean(train_lengths):.2f}, Median: {np.median(train_lengths):.2f}, Max: {np.max(train_lengths)}")
print(f"Test set - Mean: {np.mean(test_lengths):.2f}, Median: {np.median(test_lengths):.2f}, Max: {np.max(test_lengths)}")

## 3. Text Preprocessing

Text preprocessing is a crucial step for any NLP task. In this section, we'll prepare our text data for neural network training by applying various preprocessing techniques.

In [ ]:
# Define a text preprocessing class that can be reused
class TextPreprocessor:
    def __init__(self, remove_stopwords=True, stemming=False, lemmatization=True):
        self.remove_stopwords = remove_stopwords
        self.stemming = stemming
        self.lemmatization = lemmatization
        
        if self.remove_stopwords:
            self.stop_words = set(stopwords.words('english'))
        
        if self.stemming:
            self.stemmer = PorterStemmer()
            
        if self.lemmatization:
            self.lemmatizer = WordNetLemmatizer()
    
    def clean_text(self, text):
        """Basic text cleaning"""
        # Convert to lowercase
        text = text.lower()
        
        # Remove HTML tags
        text = re.sub(r'<.*?>', '', text)
        
        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def tokenize_and_process(self, text):
        """Tokenize and process text by applying stopword removal, stemming, or lemmatization"""
        # Clean the text
        text = self.clean_text(text)
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Remove stopwords if enabled
        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in self.stop_words]
        
        # Apply stemming if enabled
        if self.stemming:
            tokens = [self.stemmer.stem(token) for token in tokens]
        
        # Apply lemmatization if enabled
        if self.lemmatization:
            tokens = [self.lemmatizer.lemmatize(token) for token in tokens]
        
        return ' '.join(tokens)

# For the IMDB dataset, we need to first decode the reviews
# Create a sample of decoded reviews for demonstration
sample_size = 5
decoded_samples = [decode_review(X_train[i]) for i in range(sample_size)]

# Initialize our preprocessor
preprocessor = TextPreprocessor(remove_stopwords=True, stemming=False, lemmatization=True)

# Process the samples
processed_samples = [preprocessor.tokenize_and_process(text) for text in decoded_samples]

# Display before and after preprocessing
for i in range(sample_size):
    print(f"\n--- Sample {i+1} ---")
    print("Original:")
    print(decoded_samples[i][:200] + "..." if len(decoded_samples[i]) > 200 else decoded_samples[i])
    print("\nProcessed:")
    print(processed_samples[i][:200] + "..." if len(processed_samples[i]) > 200 else processed_samples[i])

# Note: For the IMDB dataset from Keras, the data is already preprocessed and tokenized
# For a real-world scenario, we would apply our preprocessing to raw text data
print("\nNote: The IMDB dataset from Keras is already preprocessed. This demonstration shows how we would preprocess raw text.")

## 4. Feature Extraction

Now that we have our preprocessed text, we need to convert it into numerical representations that neural networks can process. We'll explore different techniques including:

1. Sequence padding
2. One-hot encoding
3. Word embeddings

In [ ]:
# Pad sequences to ensure uniform input size
# Find a reasonable max_length (we saw the distribution earlier)
max_length = 250  # Covers most reviews as seen in the length distribution

# Pad sequences
X_train_pad = pad_sequences(X_train, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test, maxlen=max_length, padding='post', truncating='post')

print(f"Padded training data shape: {X_train_pad.shape}")
print(f"Padded test data shape: {X_test_pad.shape}")

# Let's examine a padded sequence
print("\nOriginal sequence length:", len(X_train[0]))
print("Padded sequence length:", len(X_train_pad[0]))
print("First few elements of the padded sequence:", X_train_pad[0][:20])

# Visualize the padded sequence (1 = values from the original sequence, 0 = padding)
plt.figure(figsize=(12, 2))
plt.imshow([X_train_pad[0] > 0], aspect='auto', cmap='Blues')
plt.title('Padded Sequence Visualization')
plt.xlabel('Sequence Position')
plt.yticks([])
plt.colorbar(ticks=[0, 1], orientation='vertical', label='Value')
plt.show()

# For a real-world dataset with raw text, we would use feature extraction methods like:

# 1. Bag of Words / CountVectorizer (example with our processed text samples)
print("\nBag of Words Example:")
count_vectorizer = CountVectorizer(max_features=5000)
bow_features = count_vectorizer.fit_transform(processed_samples)
print(f"Shape: {bow_features.shape}")
print("Feature names (first 10):", count_vectorizer.get_feature_names_out()[:10])

# 2. TF-IDF Vectorizer
print("\nTF-IDF Example:")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_features = tfidf_vectorizer.fit_transform(processed_samples)
print(f"Shape: {tfidf_features.shape}")
print("Feature names (first 10):", tfidf_vectorizer.get_feature_names_out()[:10])

# 3. Custom Tokenizer (similar to what Keras does internally)
print("\nCustom Tokenizer Example:")
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(processed_samples)
sequences = tokenizer.texts_to_sequences(processed_samples)
word_index = tokenizer.word_index
print(f"Found {len(word_index)} unique tokens")
print("Sample sequence:", sequences[0][:10])

## 5. Building Neural Network Models

Now we'll build different neural network architectures for text classification:

1. Simple Dense (Feedforward) Network
2. Convolutional Neural Network (CNN)
3. Recurrent Neural Network (RNN) with LSTM
4. Bidirectional LSTM

Each architecture has its strengths for different text classification tasks.

In [ ]:
# Define hyperparameters
vocab_size = num_words  # 10,000 words as defined earlier
embedding_dim = 128
epochs = 10
batch_size = 128

# Convert labels to arrays
y_train_array = np.array(y_train)
y_test_array = np.array(y_test)

# Create validation split from the training data
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_pad, y_train_array, test_size=0.2, random_state=42
)

# Define callbacks for training
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, mode='max')
]

# 1. Simple Dense Network with Embedding Layer
def create_dense_model():
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        GlobalMaxPooling1D(),  # Reduce sequence to a single vector
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# 2. CNN Model
def create_cnn_model():
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# 3. LSTM Model
def create_lstm_model():
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        SpatialDropout1D(0.3),
        LSTM(100),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# 4. Bidirectional LSTM
def create_bidirectional_lstm_model():
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        SpatialDropout1D(0.3),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Build the models
print("Building models...\n")

print("1. Dense Network:")
dense_model = create_dense_model()
dense_model.summary()

print("\n2. CNN Model:")
cnn_model = create_cnn_model()
cnn_model.summary()

print("\n3. LSTM Model:")
lstm_model = create_lstm_model()
lstm_model.summary()

print("\n4. Bidirectional LSTM Model:")
bilstm_model = create_bidirectional_lstm_model()
bilstm_model.summary()

## 6. Training the Models

In this section, we'll train our neural network models. To save time, we'll train one model as an example, but in practice you would train and compare all models to select the best one.

In [ ]:
# For demonstration, let's train the CNN model which offers a good balance 
# between performance and training speed
print("Training CNN model...")

history_cnn = cnn_model.fit(
    X_train_final, y_train_final,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

# Plot the training history
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['accuracy'], label='Training Accuracy')
plt.plot(history_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# In practice, you would train all models and compare their performance
# We'll simulate having results for all models for the evaluation section

# Note: Training LSTM models can take significant time
print("\nNote: In a complete workflow, you would train all models to compare their performance.")
print("For brevity, we've only trained the CNN model in this demonstration.")

## 7. Model Evaluation

Let's evaluate our trained model on the test set and analyze its performance using various metrics.

In [ ]:
# Evaluate the CNN model on the test set
loss, accuracy = cnn_model.evaluate(X_test_pad, y_test_array, verbose=1)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Make predictions
y_pred_prob = cnn_model.predict(X_test_pad, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test_array, y_pred))

# Create confusion matrix
cm = confusion_matrix(y_test_array, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Analyze incorrect predictions
incorrect_indices = np.where(y_pred != y_test_array)[0]
print(f"\nNumber of incorrect predictions: {len(incorrect_indices)}")

# Look at some incorrect predictions
num_samples = min(5, len(incorrect_indices))
if num_samples > 0:
    print("\nSample of Incorrect Predictions:")
    for i in range(num_samples):
        idx = incorrect_indices[i]
        review = decode_review(X_test[idx])
        true_sentiment = "positive" if y_test_array[idx] == 1 else "negative"
        pred_sentiment = "positive" if y_pred[idx] == 1 else "negative"
        
        print(f"\nReview {i+1}:")
        print(f"Text: {review[:200]}...")
        print(f"True sentiment: {true_sentiment}")
        print(f"Predicted sentiment: {pred_sentiment}")
        print(f"Prediction probability: {y_pred_prob[idx][0]:.4f}")

# Simulated comparison of different models (in practice, this would be actual results)
# This is just for demonstrating visualization of model comparison
models = ['Dense', 'CNN', 'LSTM', 'Bi-LSTM']
# Simulated accuracy values - in reality, you would use actual values from trained models
accuracies = [0.82, accuracy, 0.86, 0.87]  # Using our actual CNN accuracy

plt.figure(figsize=(10, 6))
bars = plt.bar(models, accuracies, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])
plt.title('Model Comparison')
plt.xlabel('Model Architecture')
plt.ylabel('Test Accuracy')
plt.ylim(0.75, 0.90)
plt.grid(axis='y', alpha=0.3)

# Add values above bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2.,
        height + 0.005,
        f'{height:.4f}',
        ha='center', va='bottom',
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning

To improve model performance, we can tune various hyperparameters. In practice, you might use libraries like Keras Tuner or scikit-learn's GridSearchCV. Here, we'll demonstrate a manual hyperparameter tuning process.

In [ ]:
# For demonstration purposes, we'll show how to manually tune a few hyperparameters
# In practice, you'd use more systematic approaches like grid search or random search

def create_tuned_model(embedding_dim, filters, kernel_size, dropout_rate, dense_units, learning_rate):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_length),
        Conv1D(filters, kernel_size, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(dense_units, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Define a small grid of hyperparameters to try
# In a real scenario, you would try more combinations
parameter_grid = [
    # embedding_dim, filters, kernel_size, dropout_rate, dense_units, learning_rate
    (64, 64, 3, 0.4, 64, 0.001),
    (128, 128, 5, 0.5, 128, 0.001),
]

# Create a small subset for quick demonstration
X_sample, y_sample = X_train_final[:5000], y_train_final[:5000]
X_val_sample, y_val_sample = X_val[:1000], y_val[:1000]

# Track results
tuning_results = []

for i, params in enumerate(parameter_grid):
    embedding_dim, filters, kernel_size, dropout_rate, dense_units, learning_rate = params
    
    print(f"\nTraining model with hyperparameters set {i+1}:")
    print(f"Embedding dim: {embedding_dim}, Filters: {filters}, Kernel size: {kernel_size}, "
          f"Dropout: {dropout_rate}, Dense units: {dense_units}, Learning rate: {learning_rate}")
    
    # Create and train model
    model = create_tuned_model(embedding_dim, filters, kernel_size, 
                               dropout_rate, dense_units, learning_rate)
    
    # For demonstration, train for only 2 epochs on a subset of data
    history = model.fit(
        X_sample, y_sample,
        epochs=2,  # Just 2 epochs for demonstration
        batch_size=batch_size,
        validation_data=(X_val_sample, y_val_sample),
        verbose=1
    )
    
    # Evaluate
    val_loss, val_acc = model.evaluate(X_val_sample, y_val_sample, verbose=0)
    tuning_results.append({
        'params': params,
        'val_loss': val_loss,
        'val_accuracy': val_acc
    })
    
    print(f"Validation accuracy: {val_acc:.4f}, Validation loss: {val_loss:.4f}")

# Find best hyperparameters
best_result = max(tuning_results, key=lambda x: x['val_accuracy'])
print("\nBest hyperparameters:")
embedding_dim, filters, kernel_size, dropout_rate, dense_units, learning_rate = best_result['params']
print(f"Embedding dim: {embedding_dim}, Filters: {filters}, Kernel size: {kernel_size}, "
      f"Dropout: {dropout_rate}, Dense units: {dense_units}, Learning rate: {learning_rate}")
print(f"Validation accuracy: {best_result['val_accuracy']:.4f}, Validation loss: {best_result['val_loss']:.4f}")

print("\nNote: In practice, you would perform more extensive hyperparameter tuning")
print("with more parameter combinations and using techniques like GridSearchCV or Keras Tuner.")

## 9. Handling Imbalanced Classes

In many text classification tasks, class imbalance is a common challenge. Let's explore strategies to address this issue, such as class weights, over/under-sampling, and custom loss functions.

Note: The IMDB dataset is balanced (50% positive, 50% negative), so for demonstration purposes we'll simulate an imbalanced scenario.

In [ ]:
# Create an imbalanced dataset for demonstration
# We'll select 80% of class 0 and 20% of class 1 from training data
np.random.seed(42)

# Indices for each class
class_0_indices = np.where(y_train_array == 0)[0]
class_1_indices = np.where(y_train_array == 1)[0]

# Select a subset to create imbalance (80% of class 0, 20% of class 1)
n_class_0 = int(0.8 * len(class_0_indices))
n_class_1 = int(0.2 * len(class_1_indices))

# Randomly select indices
selected_class_0 = np.random.choice(class_0_indices, n_class_0, replace=False)
selected_class_1 = np.random.choice(class_1_indices, n_class_1, replace=False)

# Combine indices and create imbalanced dataset
imbalanced_indices = np.concatenate([selected_class_0, selected_class_1])
np.random.shuffle(imbalanced_indices)

X_imbalanced = X_train_pad[imbalanced_indices]
y_imbalanced = y_train_array[imbalanced_indices]

# Check class distribution
imbalanced_dist = Counter(y_imbalanced)
print("Imbalanced class distribution:")
print(imbalanced_dist)

# Visualize imbalance
plt.figure(figsize=(8, 6))
sns.countplot(x=y_imbalanced)
plt.title('Imbalanced Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks([0, 1], ['Negative (0)', 'Positive (1)'])
plt.grid(axis='y', alpha=0.3)
plt.show()

# Split imbalanced data into train and validation
X_imb_train, X_imb_val, y_imb_train, y_imb_val = train_test_split(
    X_imbalanced, y_imbalanced, test_size=0.2, random_state=42, stratify=y_imbalanced
)

print(f"Training set shape: {X_imb_train.shape}, Class distribution: {Counter(y_imb_train)}")
print(f"Validation set shape: {X_imb_val.shape}, Class distribution: {Counter(y_imb_val)}")

# Approach 1: Use class weights
# Calculate class weights inversely proportional to class frequencies
n_samples = len(y_imb_train)
n_class_0_samples = sum(y_imb_train == 0)
n_class_1_samples = sum(y_imb_train == 1)

class_weight = {
    0: n_samples / (2 * n_class_0_samples),
    1: n_samples / (2 * n_class_1_samples)
}
print("\nClass weights:")
print(class_weight)

# Create a simple model with class weights
model_weighted = create_cnn_model()

# Train with class weights (just 2 epochs for demonstration)
history_weighted = model_weighted.fit(
    X_imb_train, y_imb_train,
    epochs=2,  # Just 2 epochs for demonstration
    batch_size=batch_size,
    validation_data=(X_imb_val, y_imb_val),
    class_weight=class_weight,
    verbose=1
)

# Approach 2: Oversampling with SMOTE (Synthetic Minority Over-sampling Technique)
# Note: For large NLP datasets, SMOTE may not be practical due to memory constraints
# We'll demonstrate the concept on a small subset
print("\nApproach 2: Oversampling with SMOTE")
print("(Demonstration concept only, not executing full SMOTE for memory reasons)")

# Import SMOTE from imbalanced-learn
try:
    from imblearn.over_sampling import SMOTE
    
    # Use a small subset for demonstration
    X_small = X_imb_train[:1000]  # Feature vectors
    y_small = y_imb_train[:1000]  # Labels
    
    print(f"Original class distribution: {Counter(y_small)}")
    
    # Apply SMOTE to the small subset
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_small, y_small)
    
    print(f"After SMOTE: {Counter(y_resampled)}")
    print(f"Shape before SMOTE: {X_small.shape}, Shape after SMOTE: {X_resampled.shape}")
    
except ImportError:
    print("imblearn not installed. Install with: pip install imbalanced-learn")
    print("Demonstrating concept without executing SMOTE.")
    print("Original distribution:", Counter(y_imb_train))
    print("After SMOTE, classes would be balanced: {0: 10000, 1: 10000}")

# Approach 3: Use a focal loss function
# Focal loss puts more weight on hard examples and less on easy ones
def focal_loss(gamma=2., alpha=.25):
    def focal_loss_fixed(y_true, y_pred):
        pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
        pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
        
        # clip to prevent NaN's and Inf's
        epsilon = 1e-7
        pt_1 = tf.clip_by_value(pt_1, epsilon, 1. - epsilon)
        pt_0 = tf.clip_by_value(pt_0, epsilon, 1. - epsilon)
        
        return -K.mean(alpha * K.pow(1. - pt_1, gamma) * K.log(pt_1) + 
                      (1 - alpha) * K.pow(pt_0, gamma) * K.log(1. - pt_0))
    return focal_loss_fixed

# The full implementation would train a model with focal loss
print("\nApproach 3: Using focal loss")
print("This approach would weight hard-to-classify examples more heavily.")
print("Implementation involves custom loss function (demonstrated in code).")

print("\nNote: In practice, for imbalanced text classification you would:")
print("1. Use class weights (simplest approach)")
print("2. Apply resampling techniques appropriate for text data")
print("3. Use specialized loss functions like focal loss")
print("4. Consider ensemble methods combining multiple approaches")

## 10. Visualizing Results

Visualizations help us understand model performance and gain insights from our text classification results.

In [ ]:
# Visualize prediction confidence distribution
y_probabilities = cnn_model.predict(X_test_pad)

plt.figure(figsize=(10, 6))
sns.histplot(y_probabilities, bins=50, kde=True)
plt.axvline(0.5, color='red', linestyle='--', label='Decision boundary')
plt.title('Distribution of Prediction Probabilities')
plt.xlabel('Probability of Positive Class')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Separate probabilities by true class
y_probs_positive = y_probabilities[y_test_array == 1].flatten()
y_probs_negative = y_probabilities[y_test_array == 0].flatten()

plt.figure(figsize=(10, 6))
sns.histplot(y_probs_positive, bins=30, color='green', alpha=0.5, kde=True, label='True Positive')
sns.histplot(y_probs_negative, bins=30, color='red', alpha=0.5, kde=True, label='True Negative')
plt.axvline(0.5, color='black', linestyle='--', label='Decision boundary')
plt.title('Prediction Probabilities by True Class')
plt.xlabel('Probability of Positive Class')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# ROC Curve
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test_array, y_probabilities)
roc_auc = auc(fpr, tpr)

# Calculate Precision-Recall curve
precision, recall, _ = precision_recall_curve(y_test_array, y_probabilities)
pr_auc = average_precision_score(y_test_array, y_probabilities)

# Plot ROC and PR curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(recall, precision, lw=2, label=f'PR curve (area = {pr_auc:.2f})')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Word Cloud visualization of correctly vs. incorrectly classified reviews
try:
    from wordcloud import WordCloud
    
    # Get incorrectly classified reviews
    incorrect_indices = np.where(y_pred != y_test_array)[0]
    
    if len(incorrect_indices) > 0:
        incorrect_texts = [decode_review(X_test[i]) for i in incorrect_indices[:100]]  # Limit to 100 examples
        incorrect_text = ' '.join(incorrect_texts)
        
        # Get correctly classified reviews
        correct_indices = np.where(y_pred == y_test_array)[0]
        correct_texts = [decode_review(X_test[i]) for i in correct_indices[:100]]  # Limit to 100 examples
        correct_text = ' '.join(correct_texts)
        
        # Create word clouds
        plt.figure(figsize=(14, 6))
        
        plt.subplot(1, 2, 1)
        wordcloud = WordCloud(width=800, height=400, background_color='white', 
                              max_words=100, contour_width=3).generate(correct_text)
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.title('Words in Correctly Classified Reviews')
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        wordcloud = WordCloud(width=800, height=400, background_color='white', 
                              max_words=100, contour_width=3).generate(incorrect_text)
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.title('Words in Incorrectly Classified Reviews')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No incorrect classifications to display.")
        
except ImportError:
    print("WordCloud package not installed. Install with: pip install wordcloud")
    print("Skipping word cloud visualization.")

## 11. Deploying the Text Classification Model

In this final section, we'll save our trained model and create utility functions to use it for classifying new text examples.

In [ ]:
# Save the best model
model_path = 'text_classification_cnn_model.h5'
cnn_model.save(model_path)
print(f"Model saved to {model_path}")

# Create a text classification function that takes raw text and returns prediction
def classify_text(text, model, max_length=max_length, word_index=word_index):
    # Preprocess the text (similar to what we did earlier)
    text = text.lower()
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove non-alphanumeric characters
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize the text
    words = text.split()
    
    # Convert words to indices
    sequence = []
    for word in words:
        # Get word index, use <UNK> token index (2) if not in vocabulary
        idx = word_index.get(word, 2) + 3  # Add 3 as in the IMDB dataset
        if idx < num_words:  # Only include if within our vocabulary size
            sequence.append(idx)
    
    # Pad sequence
    padded_sequence = pad_sequences([sequence], maxlen=max_length, padding='post', truncating='post')
    
    # Make prediction
    prediction = model.predict(padded_sequence, verbose=0)[0][0]
    
    return {
        'probability': float(prediction),
        'sentiment': 'positive' if prediction > 0.5 else 'negative',
        'confidence': float(max(prediction, 1 - prediction))
    }

# Test the classification function with some examples
test_texts = [
    "This movie was amazing! The acting was superb and the plot kept me engaged throughout.",
    "Terrible film. Bad acting, boring plot, and the special effects were laughable.",
    "I'm not sure how I feel about this one. It had good and bad moments.",
    "While the premise was interesting, the execution left a lot to be desired."
]

print("\nClassifying example texts:")
for i, text in enumerate(test_texts):
    result = classify_text(text, cnn_model)
    print(f"\nExample {i+1}:")
    print(f"Text: {text}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Probability: {result['probability']:.4f}")
    print(f"Confidence: {result['confidence']:.4f}")

# Create a simple deployment example with a text input function
def sentiment_analyzer():
    print("\n=== Sentiment Analyzer ===")
    print("Enter text to analyze (or 'q' to quit)")
    
    while True:
        user_input = input("\nText: ")
        if user_input.lower() == 'q':
            print("Exiting sentiment analyzer.")
            break
        
        if not user_input.strip():
            print("Please enter some text to analyze.")
            continue
        
        result = classify_text(user_input, cnn_model)
        print(f"Sentiment: {result['sentiment']}")
        print(f"Confidence: {result['confidence']:.2f}")

# Uncomment to run the interactive analyzer
# sentiment_analyzer()

# Also save the word index for future use
import pickle
with open('imdb_word_index.pkl', 'wb') as f:
    pickle.dump(word_index, f)
print("Word index saved to 'imdb_word_index.pkl'")

print("\n=== Deployment Notes ===")
print("In a production setting, you would:")
print("1. Save both model and preprocessing components")
print("2. Create a REST API using Flask or FastAPI")
print("3. Containerize the solution with Docker")
print("4. Deploy to cloud services (AWS, GCP, Azure)")
print("5. Set up monitoring for model performance")
print("6. Implement A/B testing for model improvements")

## Conclusion

In this notebook, we've explored text classification using neural networks, covering:

1. **Data preprocessing** techniques for text data
2. **Feature extraction** methods to convert text to numerical representations
3. **Model architectures** including Dense Networks, CNNs, and LSTMs
4. **Training and evaluation** of neural network models for text classification
5. **Handling challenges** like class imbalance
6. **Visualizing results** to gain insights
7. **Deploying models** for real-world use

Text classification is a fundamental NLP task with applications in sentiment analysis, spam detection, topic categorization, and more. The techniques covered here serve as a foundation for more advanced NLP applications.

### Next Steps

1. Experiment with pre-trained word embeddings (Word2Vec, GloVe)
2. Try transformer-based models like BERT or RoBERTa
3. Apply these techniques to domain-specific text classification tasks
4. Explore multi-class and multi-label text classification